## GPU-accelerated computations

In this notebook you will see how to:

- load data using earthkit-data
- change array format of the data
- earthkit-hydro

### Components of earthkit

This tutorial uses the following earthkit components - click any logo to open the package documentation:

<div align="center">
  <br>
  <a href="https://earthkit-data.readthedocs.io/en/latest/" target="_blank" style="display:inline-block; margin: 0 15px;">
    <img src="https://raw.githubusercontent.com/ecmwf/logos/refs/heads/main/logos/earthkit/earthkit-data-light.svg" alt="earthkit-data" width="200">
  </a>
  <a href="https://earthkit-hydro.readthedocs.io/en/latest/" target="_blank" style="display:inline-block; margin: 0 15px;">
    <img src="https://raw.githubusercontent.com/ecmwf/logos/refs/heads/main/logos/earthkit/earthkit-hydro-light.svg" alt="earthkit-geo" width="200">
  </a>
  <a href="https://earthkit-plots.readthedocs.io/en/latest/" target="_blank" style="display:inline-block; margin: 0 15px;">
    <img src="https://raw.githubusercontent.com/ecmwf/logos/refs/heads/main/logos/earthkit/earthkit-plots-light.svg" alt="earthkit-plots" width="200">
  </a> 
</div>

> Note: some of the examples in this notebook require optional dependencies - the Python package **cupy**, and the package **xarray-cupy**.

In [12]:
import earthkit.data as ekd
import earthkit.hydro as ekh
import earthkit.plots as ekp

### 1. Load a large dataset

In [ ]:
ds = ekd.from_source("sample", "R06a.nc").to_xarray()
ds

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 40, lat: 2970, lon: 4530)
Coordinates:
  * lat      (lat) float64 24kB 72.24 72.22 72.21 72.19 ... 22.79 22.77 22.76
  * lon      (lon) float64 36kB -25.24 -25.23 -25.21 ... 50.21 50.22 50.24
  * time     (time) datetime64[ns] 320B 2024-11-14T06:00:00 ... 2024-11-24
Data variables:
    R06a     (time, lat, lon) float32 2GB ...
Attributes:
    history:  Wed Mar  8 12:00:44 2023: ncks -O --mk_rec_dmn time area.templa...
    NCO:      netCDF Operators version 4.9.7 (Homepage = http://nco.sf.net, C...

In [9]:
da = ds["R06a"]

### 2. Load a river network

In [ ]:
net = ekh.river_network.load("efas", "5")

River network not found in cache (/var/folders/td/yqnxcqpx39dc855vwjtv5hj40000gn/T/tmp4pmogrem_earthkit_hydro/1.3_0dc8123bbf944ff1cb86f41bc7506e891baaa990666d836fc0cf2edd503916db.joblib).
River network loaded, saving to cache (/var/folders/td/yqnxcqpx39dc855vwjtv5hj40000gn/T/tmp4pmogrem_earthkit_hydro/1.3_0dc8123bbf944ff1cb86f41bc7506e891baaa990666d836fc0cf2edd503916db.joblib).


### 3. Conduct array-level computation using numpy

In [ ]:
array_np = da.values

%timeit ekh.upstream.array.sum(net, array_np)

array([[[0.06744136, 0.06995939, 0.08337583, ...,        nan,
                nan,        nan],
        [0.0870114 , 0.17532656, 0.33490485, ...,        nan,
                nan,        nan],
        [0.11274368, 0.23353973, 0.11811192, ...,        nan,
                nan,        nan],
        ...,
        [       nan,        nan,        nan, ..., 0.        ,
         0.        , 0.        ],
        [       nan,        nan,        nan, ..., 0.        ,
         0.        , 0.        ],
        [       nan,        nan,        nan, ..., 0.        ,
         0.        , 0.        ]],

       [[1.3761792 , 1.4072698 , 1.4729967 , ...,        nan,
                nan,        nan],
        [1.5028648 , 3.0382576 , 6.013448  , ...,        nan,
                nan,        nan],
        [1.5892987 , 3.2959242 , 1.6675166 , ...,        nan,
                nan,        nan],
        ...,
        [       nan,        nan,        nan, ..., 0.        ,
         0.        , 0.        ],
        [   

### 4. Conduct same operation with cupy

In [ ]:
import cupy as cp
net_gpu = ekh.river_network.load("efas", "5").to_device(device="cuda")
array_cp = cp.array(da.values)

%timeit ekh.upstream.array.sum(net, array_np)

### 5. Xarray computation backed by numpy

In [ ]:
small_da = da.isel(time=0)
%timeit ekh.upstream.sum(net, small_da)

### 6. Xarray computation backed by cupy

In [ ]:
small_da_gpu = da.isel(time=0).cupy.as_cupy()
%timeit ekh.upstream.sum(net_gpu, small_da_gpu)

### 7. Plotting

In [ ]:
style = ekp.styles.Style(
    colors="Blues",
    levels=[0, 0.5, 1, 2, 5, 10, 50, 100, 500, 1000, 2000, 3000, 4000],
    extend="max",
)

upstream_sum = ekh.upstream.sum(net, small_da)

chart = ekp.Map()
chart.quickplot(upstream_sum, style=style)
chart.legend(label="{variable_name}")
chart.title("Upstream precipitation at {time:%H:%M on %-d %B %Y}")
chart.coastlines()
chart.gridlines()
chart.show()